# Day 6 | Lab 6.1: AWS Bedrock Production Patterns — ChatBedrock + OpenSearch RAG

**Duration:** ~1.5 hours

**Scenario.** A regional bank exposes uploaded financial documents (Q3 financial report, commercial-loan template, AML compliance manual) as a Q&A bot for relationship managers. We use **AWS Bedrock** end-to-end: Titan Embeddings for vectors, OpenSearch Serverless as the vector store, Claude Sonnet 4.5 (via `ChatBedrock`) for generation.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Wire `ChatBedrock(model_id='anthropic.claude-sonnet-4-5-...')` into a production LCEL RAG chain.
2. Index documents into Amazon OpenSearch with Titan Text Embeddings v2 (1024-dim, cosine similarity).
3. Compose a `RunnableParallel` chain that returns both the answer and the source documents.
4. Add multi-turn memory using `RunnableWithMessageHistory` (replaces the source notebook's hand-rolled history list).
5. Compare the same model called via direct Anthropic API (Lab 6.1) vs Bedrock — latency, cost, IAM/VPC trade-offs.

**Tools.** AWS Bedrock · `langchain-aws` `ChatBedrock` + `BedrockEmbeddings` · OpenSearch Serverless · `langchain-text-splitters` · LCEL · `RunnableWithMessageHistory`

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-aws>=0.2' 'langchain-text-splitters' 'langchain-community' boto3 opensearch-py requests-aws4auth python-dotenv pypdf


In [3]:
%pip install requests-aws4auth -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install -U opensearch-py -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [1]:
import os
import json
import time
from typing import List, Dict

import boto3
from requests_aws4auth import AWS4Auth

from opensearchpy import OpenSearch, RequestsHttpConnection

from langchain_aws import BedrockEmbeddings, ChatBedrock
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableParallel, RunnablePassthrough, RunnableLambda,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

print('✅ Imports OK')


✅ Imports OK


## 3. AWS Credentials

Local-venv pattern. Set `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` (and your `OPENSEARCH_ENDPOINT`) in your `.env` file or shell environment before running.


In [3]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION', 'OPENSEARCH_ENDPOINT']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


AWS_ACCESS_KEY_ID: ✅ Loaded
AWS_SECRET_ACCESS_KEY: ✅ Loaded
AWS_DEFAULT_REGION: ✅ Loaded
OPENSEARCH_ENDPOINT: ✅ Loaded


## 4. Initialise AWS Clients (Bedrock + OpenSearch)

In [4]:
aws_region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
opensearch_endpoint = os.environ.get('OPENSEARCH_ENDPOINT', '')  # e.g. xxxx.us-east-1.es.amazonaws.com

session = boto3.Session(region_name=aws_region)
credentials = session.get_credentials()

bedrock_runtime = session.client('bedrock-runtime', region_name=aws_region)
bedrock         = session.client('bedrock',         region_name=aws_region)

awsauth = AWS4Auth(
    credentials.access_key, credentials.secret_key, aws_region, 'es',
    session_token=credentials.token,
)

opensearch_client = OpenSearch(
    hosts=[{'host': opensearch_endpoint, 'port': 443}],
    http_auth=awsauth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=30,
)

try:
    info = opensearch_client.info()
    print(f"✅ Connected to OpenSearch: {info['cluster_name']} (v{info['version']['number']})")
except Exception as e:
    print(f'⚠  Could not connect to OpenSearch — {e}')


✅ Connected to OpenSearch: 496750522080:eclerx (v3.3.0)


## 5. Sample Financial Documents

Three synthetic documents: a Q3 financial report, a commercial-loan agreement template, and an AML compliance manual. These mimic the kind of files a relationship manager would query in production.


In [5]:
# Sample financial documents (simulate uploaded files)
financial_documents = [
    {
        "filename": "Q3_2024_Financial_Report.txt",
        "content": """
QUARTERLY FINANCIAL REPORT - Q3 2024
Federal Reserve Bank - Regional Analysis

EXECUTIVE SUMMARY
The third quarter of 2024 showed continued resilience in the banking sector despite
persistent inflation concerns. Key metrics indicate stable growth with cautious optimism
for Q4 performance.

KEY FINANCIAL METRICS:

Total Assets: $2.8 trillion (up 3.2% YoY)
Total Deposits: $2.1 trillion (up 2.8% YoY)
Total Loans: $1.6 trillion (up 4.1% YoY)
Net Interest Income: $42.5 billion (up 5.3% YoY)
Return on Assets (ROA): 1.12%
Return on Equity (ROE): 11.8%

LOAN PORTFOLIO COMPOSITION:

Commercial & Industrial Loans: $580 billion (36.25%)
- Strong performance in technology sector
- Increased activity in renewable energy financing
- Default rate: 0.95% (below industry average of 1.2%)

Residential Mortgages: $480 billion (30%)
- Average loan size: $385,000
- 30-year fixed rate average: 6.8%
- Refinancing activity down 45% compared to Q3 2023
- Delinquency rate: 1.8% (stable)

Commercial Real Estate: $320 billion (20%)
- Office space segment shows weakness (10% decline)
- Industrial and logistics strong (15% growth)
- Multifamily housing stable
- Overall default rate: 2.1%

Consumer Loans: $160 billion (10%)
- Auto loans: $95 billion
- Credit cards: $45 billion
- Personal loans: $20 billion
- Average APR: 11.5%
- Charge-off rate: 1.4%

Small Business Loans: $60 billion (3.75%)
- SBA 7(a) loans: $35 billion
- Microloans: $8 billion
- Strong demand in healthcare and professional services
- Default rate: 1.8%

DEPOSIT ANALYSIS:

Non-interest bearing deposits: $630 billion (30%)
Interest bearing deposits: $1,470 billion (70%)

Average interest rates paid:
- Savings accounts: 2.5%
- Money market accounts: 3.8%
- Certificates of Deposit (12-month): 5.0%
- Checking accounts: 0.5%

CAPITAL ADEQUACY:

Tier 1 Capital Ratio: 12.5% (well above regulatory minimum of 6%)
Total Capital Ratio: 15.2% (regulatory minimum: 8%)
Leverage Ratio: 8.8%
Common Equity Tier 1 (CET1): 11.3%

CREDIT QUALITY:

Non-performing assets: $18.5 billion (0.66% of total assets)
Allowance for loan losses: $28.2 billion (1.76% of total loans)
Net charge-offs: $2.1 billion (0.13% of average loans)
Coverage ratio: 152% (allowance to non-performing loans)

OPERATIONAL EFFICIENCY:

Efficiency Ratio: 58.5% (target: <60%)
Cost-to-Income Ratio: 62.3%
Employees: 42,500 (down 2.1% YoY due to automation)
Branches: 1,850 (down 5.2% YoY)
Digital banking users: 8.2 million (up 12% YoY)

RISK MANAGEMENT:

Value at Risk (VaR) 95%: $125 million
Liquidity Coverage Ratio (LCR): 135% (regulatory minimum: 100%)
Net Stable Funding Ratio (NSFR): 118%
Stress test results: Passed all scenarios

MARKET CONDITIONS:

Federal Funds Rate: 5.25% - 5.50% (unchanged from Q2)
10-year Treasury Yield: 4.35% (average for quarter)
Unemployment Rate: 3.8%
GDP Growth: 2.1% (annualized)
Inflation (CPI): 3.2% (trending down from 3.7% in Q2)

OUTLOOK FOR Q4 2024:

Expected loan growth: 3-4%
Net interest margin forecast: 3.2% (slight compression)
Credit quality: Stable to slightly improving
Technology investments: $450 million planned
Focus areas: AI/ML for fraud detection, digital banking enhancements

REGULATORY COMPLIANCE:

All regulatory capital requirements exceeded
Zero material findings in recent examinations
Anti-money laundering (AML) programs enhanced
Cybersecurity investments: $125 million in Q3
CECL (Current Expected Credit Loss) fully implemented

CONCLUSION:

The banking sector demonstrated resilience in Q3 2024. Strong capital positions,
improving credit quality, and strategic investments in technology position institutions
well for continued stability. Management remains cautiously optimistic about Q4,
monitoring economic indicators closely.
        """
    },
    {
        "filename": "Commercial_Loan_Agreement_Template.txt",
        "content": """
COMMERCIAL LOAN AGREEMENT

This Commercial Loan Agreement ("Agreement") is entered into as of [DATE],
between [BANK NAME] ("Lender") and [BORROWER NAME] ("Borrower").

ARTICLE 1: LOAN TERMS

1.1 LOAN AMOUNT
The Lender agrees to provide a commercial loan to the Borrower in the principal
amount of $[AMOUNT] ("Loan Amount").

1.2 INTEREST RATE
The Loan shall bear interest at a rate of [RATE]% per annum. The interest rate is
[fixed/variable] for the term of the Loan.

For variable rate loans:
- Base rate: Wall Street Journal Prime Rate
- Margin: [X]% over prime
- Floor rate: [Y]%
- Ceiling rate: [Z]%
- Rate adjustment frequency: Quarterly

1.3 LOAN TERM
The term of this Loan shall be [NUMBER] years, commencing on [START DATE] and
ending on [MATURITY DATE].

1.4 REPAYMENT SCHEDULE
The Borrower shall repay the Loan in [NUMBER] monthly installments of principal
and interest, with the first payment due on [DATE].

Monthly Payment Calculation:
- Principal and Interest: $[AMOUNT]
- Payment frequency: Monthly
- Due date: [DAY] of each month
- Late payment grace period: 10 days
- Late payment fee: 5% of payment amount or $100, whichever is greater

ARTICLE 2: USE OF PROCEEDS

2.1 PERMITTED USES
The Loan proceeds shall be used exclusively for the following business purposes:
- Working capital
- Equipment purchase
- Real estate acquisition or improvement
- Business expansion
- Refinancing existing business debt

2.2 PROHIBITED USES
Loan proceeds shall NOT be used for:
- Personal expenses
- Speculative investments
- Illegal activities
- Purchase of securities
- Distribution to owners (unless approved)

ARTICLE 3: COLLATERAL

3.1 SECURITY INTEREST
To secure the Loan, the Borrower grants the Lender a first priority security
interest in the following collateral:

Real Property:
- Address: [PROPERTY ADDRESS]
- Appraised value: $[VALUE]
- Loan-to-Value ratio: [X]%

Equipment:
- Description: [EQUIPMENT LIST]
- Estimated value: $[VALUE]
- UCC filing required

Accounts Receivable:
- Blanket lien on all accounts
- Advance rate: 80% of eligible receivables

Inventory:
- Blanket lien on all inventory
- Advance rate: 50% of eligible inventory

3.2 COLLATERAL INSURANCE
The Borrower shall maintain insurance on all collateral:
- Property insurance: Replacement value
- Liability insurance: Minimum $2,000,000
- Business interruption: 12 months coverage
- Lender named as loss payee and additional insured

3.3 PERSONAL GUARANTEE
Required for all owners with 20% or more equity stake.
- Joint and several liability
- Unlimited guarantee
- Spouse signature required in community property states

ARTICLE 4: FINANCIAL COVENANTS

4.1 DEBT SERVICE COVERAGE RATIO (DSCR)
Minimum DSCR of 1.25:1, tested quarterly
Calculation: (Net Operating Income) / (Total Debt Service)

4.2 CURRENT RATIO
Minimum current ratio of 1.5:1, tested quarterly
Calculation: (Current Assets) / (Current Liabilities)

4.3 DEBT-TO-EQUITY RATIO
Maximum debt-to-equity ratio of 3.0:1, tested annually
Calculation: (Total Liabilities) / (Total Equity)

4.4 MINIMUM WORKING CAPITAL
Maintain minimum working capital of $[AMOUNT] at all times
Working Capital = Current Assets - Current Liabilities

ARTICLE 5: BORROWER REPRESENTATIONS

The Borrower represents and warrants:
- Legal entity in good standing
- Authority to enter into this Agreement
- No existing defaults on other obligations
- Financial statements are true and accurate
- No pending or threatened litigation (material)
- Compliance with all laws and regulations
- No material adverse changes since application

ARTICLE 6: REPORTING REQUIREMENTS

6.1 ANNUAL REQUIREMENTS
Within 90 days of fiscal year end:
- Audited financial statements (if revenue >$5M)
- Tax returns (business and personal guarantors)
- Updated business plan and projections

6.2 QUARTERLY REQUIREMENTS
Within 45 days of quarter end:
- Unaudited financial statements
- Accounts receivable aging
- Accounts payable aging
- Covenant compliance certificate

6.3 SPECIAL REPORTING
Immediate notification required for:
- Defaults on other obligations
- Material litigation
- Material adverse changes
- Change in ownership
- Loss of major customer or supplier

ARTICLE 7: EVENTS OF DEFAULT

7.1 DEFAULT EVENTS
The following constitute Events of Default:
- Failure to make payment within 10 days of due date
- Breach of any covenant
- Material misrepresentation
- Bankruptcy or insolvency
- Material adverse change
- Cross-default on other obligations
- Failure to maintain insurance
- Unauthorized sale or transfer of collateral

7.2 REMEDIES
Upon Event of Default, Lender may:
- Declare entire balance immediately due
- Increase interest rate by 2% (default rate)
- Foreclose on collateral
- Appoint receiver
- Exercise all rights under UCC
- Pursue collection from guarantors

ARTICLE 8: PREPAYMENT

8.1 PREPAYMENT RIGHTS
Borrower may prepay without penalty after [X] years.

8.2 PREPAYMENT PENALTY
If prepaid before [X] years:
- Years 1-2: 3% of outstanding balance
- Years 3-4: 2% of outstanding balance
- Year 5+: 1% of outstanding balance

ARTICLE 9: FEES AND COSTS

9.1 ORIGINATION FEE
[X]% of loan amount, due at closing

9.2 ANNUAL REVIEW FEE
$[AMOUNT] per year, payable on anniversary date

9.3 OTHER FEES
- Wire transfer fee: $35
- Returned check fee: $50
- Late payment fee: 5% or $100 (whichever greater)
- Document preparation: $500
- Appraisal fees: Actual cost
- Environmental assessment: Actual cost
- Legal fees: Actual cost

ARTICLE 10: MISCELLANEOUS

10.1 GOVERNING LAW
This Agreement shall be governed by the laws of [STATE].

10.2 ENTIRE AGREEMENT
This Agreement constitutes the entire agreement between parties.

10.3 AMENDMENTS
Amendments must be in writing and signed by both parties.

10.4 NOTICES
All notices must be in writing and sent to addresses on file.

10.5 SEVERABILITY
If any provision is invalid, the remainder shall continue in effect.
        """
    },
    {
        "filename": "AML_Compliance_Manual_2024.txt",
        "content": """
ANTI-MONEY LAUNDERING (AML) COMPLIANCE MANUAL
Updated: October 2024

CHAPTER 1: OVERVIEW AND POLICY

1.1 PURPOSE
This manual establishes the bank's Anti-Money Laundering (AML) program in compliance
with the Bank Secrecy Act (BSA), USA PATRIOT Act, and FinCEN regulations.

1.2 SCOPE
This policy applies to all employees, officers, directors, and third parties acting
on behalf of the bank.

1.3 REGULATORY FRAMEWORK
Key regulations:
- Bank Secrecy Act (BSA)
- USA PATRIOT Act Section 326 (Customer Identification Program)
- USA PATRIOT Act Section 312 (Due Diligence for Correspondent Accounts)
- USA PATRIOT Act Section 314(a) and (b) (Information Sharing)
- FinCEN Regulations
- OFAC Sanctions Programs

CHAPTER 2: CUSTOMER IDENTIFICATION PROGRAM (CIP)

2.1 CIP REQUIREMENTS
Before opening an account, obtain and verify:

For Individuals:
- Full legal name
- Date of birth
- Residential address (P.O. Box not acceptable as sole address)
- Taxpayer Identification Number (SSN or ITIN)
- Government-issued photo ID

For Business Entities:
- Legal business name
- Business address
- Employer Identification Number (EIN)
- Business formation documents
- Beneficial ownership information (individuals with 25%+ ownership)
- Authorized signers

2.2 VERIFICATION METHODS
Documentary Verification:
- Government-issued ID (driver's license, passport)
- Utility bills for address verification
- Business formation documents

Non-Documentary Verification:
- Credit bureau reports
- Public database checks
- Third-party verification services

2.3 VERIFICATION TIMEFRAME
- Standard accounts: Verify before account opening
- Exception procedures: Verify within 30 days (documented reason required)
- Account restrictions apply until verification complete

CHAPTER 3: CUSTOMER DUE DILIGENCE (CDD)

3.1 BENEFICIAL OWNERSHIP RULE
For legal entity customers, identify:

Ownership Prong:
- Each individual owning 25% or more of equity
- Name, date of birth, address, SSN/ITIN

Control Prong:
- One individual with significant control (CEO, CFO, COO, President)
- If no single person, identify managing member or partner

3.2 CUSTOMER RISK RATING

Low Risk:
- Individual accounts <$50,000
- Established relationship >5 years
- Salary/employment income only
- Local customer

Medium Risk:
- Business accounts
- High-value individual accounts ($50K-$500K)
- International wire activity
- Cash-intensive businesses (below threshold)

High Risk:
- Cash-intensive businesses (ATMs, check cashing, money transmitters)
- Foreign correspondent banking
- Non-resident alien accounts
- Politically Exposed Persons (PEPs)
- High-value accounts (>$500K)
- Countries of concern (OFAC list)

3.3 ENHANCED DUE DILIGENCE (EDD)
Required for high-risk customers:
- Understanding of business and anticipated activity
- Source of funds and source of wealth
- Purpose of account
- Enhanced monitoring
- Senior management approval
- Periodic review (at least annually)

CHAPTER 4: SUSPICIOUS ACTIVITY MONITORING

4.1 RED FLAGS

Transaction Red Flags:
- Structuring (multiple transactions just under $10,000)
- Transactions inconsistent with business
- Rapid movement of funds
- Frequent wire transfers to high-risk countries
- Round-dollar amounts (large)
- Lack of economic purpose

Customer Behavior Red Flags:
- Reluctance to provide information
- Provides false or suspicious information
- Unusual concern about compliance questions
- Lacks knowledge of transaction details
- Attempts to avoid reporting thresholds

Account Activity Red Flags:
- Dormant account suddenly active
- Activity inconsistent with stated business
- Multiple accounts with unexplained links
- Frequent deposits followed by immediate withdrawals
- Use of multiple accounts at different branches

4.2 TRANSACTION MONITORING

Automated Monitoring:
- Real-time transaction screening
- Daily batch processing
- Threshold-based alerts
- Behavior-based analytics
- Pattern recognition algorithms

Manual Monitoring:
- Teller observations
- Relationship manager referrals
- Customer service reports
- Branch manager reviews

CHAPTER 5: SUSPICIOUS ACTIVITY REPORTING (SAR)

5.1 SAR FILING REQUIREMENTS

File SAR when:
- Transaction >$5,000 involves potential money laundering or BSA violation
- Transaction >$5,000 designed to evade BSA requirements
- Transaction >$5,000 lacks business or lawful purpose
- Computer intrusion or breach (any amount)

5.2 SAR FILING TIMELINE
- File within 30 days of initial detection
- Extension to 60 days if suspect not identified
- No extension beyond 60 days permitted

5.3 SAR CONFIDENTIALITY
CRITICAL: DO NOT disclose SAR filing to:
- Customer or subject
- Any person involved in transaction
- Anyone outside bank (except regulators, law enforcement)

Violation penalty: Criminal and civil liability

5.4 SAR DOCUMENTATION
Maintain for 5 years:
- Copy of SAR filed
- Supporting documentation
- Investigation notes
- Transaction records

CHAPTER 6: CURRENCY TRANSACTION REPORTING (CTR)

6.1 CTR REQUIREMENTS
File CTR for:
- Cash transactions >$10,000 in single day
- Multiple transactions that aggregate >$10,000
- Includes deposits, withdrawals, exchanges, transfers

6.2 CTR EXEMPTIONS

Eligible for Exemption:
- Banks and financial institutions
- Government entities
- Listed public companies
- Non-listed businesses (after 2 months relationship and due diligence)

NOT Eligible:
- Individuals (never exempt)
- Non-account holders
- Businesses <2 months relationship

6.3 CTR FILING
- File electronically through BSA E-Filing System
- Deadline: 15 days after transaction
- Include all required information
- Maintain copy for 5 years

CHAPTER 7: OFAC COMPLIANCE

7.1 OFAC SCREENING

Screen against OFAC lists:
- Specially Designated Nationals (SDN)
- Blocked Persons List
- Sectoral Sanctions Identifications List
- Foreign Sanctions Evaders List
- Other OFAC programs

7.2 SCREENING POINTS
- New account opening
- Wire transfers (both incoming and outgoing)
- Trade finance transactions
- Periodic customer rescreening (quarterly minimum)

7.3 OFAC MATCH PROCEDURES

If potential match:
1. STOP transaction immediately
2. Do NOT notify customer
3. Review match quality (name, address, DOB, etc.)
4. Escalate to OFAC Compliance Officer
5. If true match: Block assets, file report within 10 days
6. If false positive: Document and release

CHAPTER 8: TRAINING REQUIREMENTS

8.1 MANDATORY TRAINING

All Employees:
- Initial training within 30 days of hire
- Annual refresher training
- Minimum 2 hours per year

High-Risk Roles (additional):
- Tellers: Cash handling, structuring detection
- Wire operators: OFAC screening, sanctions
- Customer service: CIP, CDD procedures
- Compliance: Advanced SAR writing, investigation

8.2 TRAINING TOPICS
- BSA/AML overview
- Customer identification
- Suspicious activity detection
- SAR and CTR filing
- OFAC compliance
- Red flags and indicators
- Case studies
- Consequences of non-compliance

CHAPTER 9: INDEPENDENT TESTING

9.1 AUDIT REQUIREMENTS
- Annual independent audit required
- Performed by qualified auditor
- Cover all aspects of AML program
- Report to Board and senior management

9.2 AUDIT SCOPE
- Program effectiveness
- Policy compliance
- CIP/CDD implementation
- Transaction monitoring
- SAR/CTR filing accuracy and timeliness
- OFAC compliance
- Training adequacy
- Record retention

CHAPTER 10: PENALTIES AND CONSEQUENCES

10.1 REGULATORY PENALTIES
- Civil penalties up to $250,000 per violation
- Criminal penalties up to $500,000 per violation
- Imprisonment up to 10 years
- Consent orders and enforcement actions
- Regulatory restrictions

10.2 INSTITUTIONAL CONSEQUENCES
- Reputational damage
- Loss of banking charter
- Federal Deposit Insurance Corporation (FDIC) sanctions
- Mandatory compliance program enhancements
- Increased regulatory oversight
        """
    }
]

print(f"✅ Created {len(financial_documents)} financial documents")
print("\nDocuments to be uploaded:")
for i, doc in enumerate(financial_documents, 1):
    words = len(doc['content'].split())
    print(f"  {i}. {doc['filename']} ({words:,} words)")

✅ Created 3 financial documents

Documents to be uploaded:
  1. Q3_2024_Financial_Report.txt (528 words)
  2. Commercial_Loan_Agreement_Template.txt (899 words)
  3. AML_Compliance_Manual_2024.txt (1,115 words)


## 6. Persist Documents to Disk

In [6]:
docs_dir = 'uploaded_documents'
os.makedirs(docs_dir, exist_ok=True)
for doc in financial_documents:
    with open(os.path.join(docs_dir, doc['filename']), 'w', encoding='utf-8') as f:
        f.write(doc['content'])
print(f"✅ Wrote {len(financial_documents)} documents to ./{docs_dir}")


✅ Wrote 3 documents to ./uploaded_documents


## 7. Load and Chunk

In [7]:
loader = DirectoryLoader(docs_dir, glob='**/*.txt', loader_cls=TextLoader)
documents = loader.load()
print(f'Loaded {len(documents)} documents')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
chunks = splitter.split_documents(documents)
for i, c in enumerate(chunks):
    c.metadata['chunk_id'] = i
    c.metadata['filename'] = os.path.basename(c.metadata.get('source', ''))
print(f'Produced {len(chunks)} chunks')


Loaded 3 documents
Produced 23 chunks


## 8. Bedrock Embeddings — Amazon Titan Text v2

In [8]:
embeddings = BedrockEmbeddings(
    model_id='amazon.titan-embed-text-v2:0',
    client=bedrock_runtime,
    region_name=aws_region,
)
test_vec = embeddings.embed_query('What are the regulatory penalties per violation?')
print(f'✅ Titan v2 embeddings — dim={len(test_vec)}')


✅ Titan v2 embeddings — dim=1024


## 9. Create OpenSearch Index

In [ ]:
index_name = os.environ.get('OPENSEARCH_INDEX', 'prashant-rag')

if opensearch_client.indices.exists(index=index_name):
    print(f'⚠  Index {index_name!r} exists — deleting for a clean run.')
    opensearch_client.indices.delete(index=index_name)

index_body = {
    'settings': {'index': {'knn': True, 'number_of_shards': 2, 'number_of_replicas': 1}},
    'mappings': {'properties': {
        'text': {'type': 'text'},
        'vector_field': {
            'type': 'knn_vector', 'dimension': 1024,
            'method': {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': 'lucene',
                       'parameters': {'ef_construction': 128, 'm': 24}},
        },
        'metadata': {'properties': {
            'filename': {'type': 'keyword'}, 'source': {'type': 'keyword'},
            'chunk_id': {'type': 'integer'},
        }},
    }},
}
opensearch_client.indices.create(index=index_name, body=index_body)
print(f'✅ Created index → {index_name}')


✅ Created index → m6-rag-claude


## 10. Index Document Chunks

In [10]:
vectorstore = OpenSearchVectorSearch(
    opensearch_url=f'https://{opensearch_endpoint}',
    index_name=index_name,
    embedding_function=embeddings,
    http_auth=awsauth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=30,
)

batch = 10
for i in range(0, len(chunks), batch):
    vectorstore.add_documents(chunks[i:i+batch])
    progress = min(i + batch, len(chunks))
    print(f'  Indexed {progress}/{len(chunks)}')
    time.sleep(0.5)
print('✅ Indexing complete')


  Indexed 10/23
  Indexed 20/23
  Indexed 23/23
✅ Indexing complete


## 11. Sanity-Check Retrieval

In [22]:
test_query = 'What are the regulatory penalties per violation?'
hits = vectorstore.similarity_search(test_query, k=3)
for i, h in enumerate(hits, 1):
    print(f"\n[{i}] {h.metadata.get('filename')}")
    print(h.page_content[:250], '...')



[1] AML_Compliance_Manual_2024.txt
CHAPTER 9: INDEPENDENT TESTING

9.1 AUDIT REQUIREMENTS
- Annual independent audit required
- Performed by qualified auditor
- Cover all aspects of AML program
- Report to Board and senior management

9.2 AUDIT SCOPE
- Program effectiveness
- Policy c ...

[2] Q3_2024_Financial_Report.txt
RISK MANAGEMENT:

Value at Risk (VaR) 95%: $125 million
Liquidity Coverage Ratio (LCR): 135% (regulatory minimum: 100%)
Net Stable Funding Ratio (NSFR): 118%
Stress test results: Passed all scenarios

MARKET CONDITIONS:

Federal Funds Rate: 5.25% - 5 ...

[3] AML_Compliance_Manual_2024.txt
Customer Behavior Red Flags:
- Reluctance to provide information
- Provides false or suspicious information
- Unusual concern about compliance questions
- Lacks knowledge of transaction details
- Attempts to avoid reporting thresholds

Account Activi ...


## 12. Initialise the LLM — Claude Sonnet 4.5 on Bedrock

**Refactor note.** The source notebook used `amazon.nova-pro-v1:0`. Per the Module 6 plan we replace it with `anthropic.claude-sonnet-4-5-...` on Bedrock — same `ChatBedrock` interface, same LCEL chain, dramatically better instruction-following on regulated banking content.


In [14]:
llm = ChatBedrock(
    model_id='amazon.nova-pro-v1:0',
    client=bedrock_runtime,
    region_name=aws_region,
    model_kwargs={
        'max_tokens': 2048,
        'temperature': 0.1,    # Low for factual Q&A
        'top_p': 0.9,
    },
)
ping = llm.invoke("Reply with exactly: 'Bedrock + Claude ready'")
print('✅', ping.content)


✅ Bedrock + Claude ready


## 13. Build the LCEL RAG Chain

In [15]:
qa_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a financial-services analyst. Use ONLY the retrieved context to answer. '
     'Quote specific numbers, percentages, and dates when present. If the answer is not in '
     'the context, say so explicitly. Mention which document(s) you used.'),
    ('human',
     'Question: {question}\n\nContext:\n{context}\n\nAnswer:')
])

def format_docs(docs: List[Document]) -> str:
    parts = []
    for d in docs:
        parts.append(f"Source: {d.metadata.get('filename', 'unknown')}\n{d.page_content}")
    return '\n\n---\n\n'.join(parts)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

rag_chain = RunnableParallel(
    answer=(
        {'context':  retriever | RunnableLambda(format_docs),
         'question': RunnablePassthrough()}
        | qa_prompt | llm | StrOutputParser()
    ),
    source_documents=retriever,
)
print('✅ LCEL RAG chain ready (retriever | prompt | Claude | parser)')


✅ LCEL RAG chain ready (retriever | prompt | Claude | parser)


## 14. Test the Q&A System

In [16]:
def ask(question: str) -> None:
    print('=' * 80)
    print(f'💬 {question}')
    print('=' * 80)
    t0 = time.time()
    out = rag_chain.invoke(question)
    dt = time.time() - t0
    print(f"\n🤖 {out['answer']}")
    print(f'\n⏱ {dt:.2f}s · {len(out["source_documents"])} sources')
    for i, d in enumerate(out['source_documents'], 1):
        print(f"  [{i}] {d.metadata.get('filename')} (chunk {d.metadata.get('chunk_id')})")

for q in [
    'What were the total assets in Q3 2024?',
    'What are the financial covenants in a commercial loan?',
    'What is the SAR filing timeline under BSA?',
]:
    ask(q)


💬 What were the total assets in Q3 2024?

🤖 The total assets in Q3 2024 were $2.8 trillion. (Source: Q3_2024_Financial_Report.txt)

⏱ 2.61s · 3 sources
  [1] Q3_2024_Financial_Report.txt (chunk 18)
  [2] Q3_2024_Financial_Report.txt (chunk 21)
  [3] Q3_2024_Financial_Report.txt (chunk 19)
💬 What are the financial covenants in a commercial loan?

🤖 Based on the provided context from the "Commercial_Loan_Agreement_Template.txt," the financial covenants in the commercial loan are as follows:

1. **Debt Service Coverage Ratio (DSCR)**: 
   - Minimum DSCR of 1.25:1, tested quarterly.
   - Calculation: (Net Operating Income) / (Total Debt Service).

2. **Current Ratio**: 
   - Minimum current ratio of 1.5:1, tested quarterly.
   - Calculation: (Current Assets) / (Current Liabilities).

3. **Debt-to-Equity Ratio**: 
   - Maximum debt-to-equity ratio of 3.0:1, tested annually.
   - Calculation: (Total Liabilities) / (Total Equity).

4. **Minimum Working Capital**: 
   - Maintain minimum workin

## 15. Conversational RAG with `RunnableWithMessageHistory`

**Refactor note.** The source notebook tracked chat history with a hand-rolled Python list (`chat_history: List`). Per CLAUDE.md §2.1 (deprecated APIs), we use the modern LCEL pattern: wrap the chain with `RunnableWithMessageHistory` and let LangChain manage per-session history via `InMemoryChatMessageHistory`. Storage is now swappable to Postgres/Redis without touching the chain.


In [20]:
# 1. The conversational chain — note the MessagesPlaceholder for history.
from operator import itemgetter

conv_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a financial-services analyst. Use ONLY the retrieved context plus the conversation history '
     'to answer. If the answer is not in the context, say so. Cite documents.'),
    MessagesPlaceholder(variable_name='history'),
    ('human',
     'Question: {question}\n\nContext:\n{context}\n\nAnswer:')
])

# RunnableWithMessageHistory passes a dict {'question': ..., 'history': [...]} into the chain.
# Use .assign() so 'question' and 'history' flow through to the prompt, and build 'context' by
# pulling ONLY the question string out of the dict before handing it to the retriever — otherwise
# the retriever embeds the whole dict and BedrockEmbeddings raises AttributeError on .replace().
conv_chain = (
    RunnablePassthrough.assign(
        context=itemgetter('question') | retriever | RunnableLambda(format_docs),
    )
    | conv_prompt | llm | StrOutputParser()
)

# 2. Per-session history store — InMemory now, swap to RedisChatMessageHistory in prod.
store: dict[str, InMemoryChatMessageHistory] = {}
def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. Wrap.
conv_with_history = RunnableWithMessageHistory(
    conv_chain,
    get_history,
    input_messages_key='question',
    history_messages_key='history',
)
print('✅ Conversational RAG wrapped with RunnableWithMessageHistory')


✅ Conversational RAG wrapped with RunnableWithMessageHistory


In [21]:
session_id = 'rm-priya-001'
config = {'configurable': {'session_id': session_id}}

for q in [
    'What were the total assets in Q3 2024?',
    'How does that compare to the loan portfolio?',  # follow-up — needs prior turn
    'And what fraction is commercial real estate?',
]:
    print(f'\n💬 {q}')
    answer = conv_with_history.invoke({'question': q}, config=config)
    print(f'🤖 {answer}')

print(f'\n📜 Stored {len(store[session_id].messages)} messages in this session.')



💬 What were the total assets in Q3 2024?
🤖 The total assets in Q3 2024 were $2.8 trillion. This figure represents a 3.2% increase year-over-year (YoY). 

**Source:** Q3_2024_Financial_Report.txt

💬 How does that compare to the loan portfolio?
🤖 The total assets in Q3 2024 were $2.8 trillion, while the total loan portfolio was $1.6 trillion. This means that the loan portfolio represented approximately 57.1% of the total assets.

**Source:** Q3_2024_Financial_Report.txt

💬 And what fraction is commercial real estate?
🤖 Commercial real estate represents 20% of the total loan portfolio, which amounts to $320 billion.

**Source:** Q3_2024_Financial_Report.txt

📜 Stored 6 messages in this session.


## 16. Cost Monitoring (Claude on Bedrock)

In [19]:
def estimate_rag_costs(num_queries: int, avg_in_tok: int = 2000, avg_out_tok: int = 500) -> Dict:
    '''Bedrock pricing (us-east-1, indicative). Verify against current AWS pricing page.'''
    titan_per_1k        = 0.00002   # Titan Embed v2 input
    claude_in_per_1m    = 3.00      # Sonnet 4.5 input  ($/1M tok)
    claude_out_per_1m   = 15.00     # Sonnet 4.5 output ($/1M tok)
    embed_cost = (num_queries * avg_in_tok / 1000) * titan_per_1k
    in_cost    = (num_queries * avg_in_tok  / 1_000_000) * claude_in_per_1m
    out_cost   = (num_queries * avg_out_tok / 1_000_000) * claude_out_per_1m
    total = embed_cost + in_cost + out_cost
    return {'queries': num_queries, 'embed': embed_cost, 'llm_in': in_cost, 'llm_out': out_cost,
            'total': total, 'per_query': total / num_queries}

for s in [{'name':'Light','q':100},{'name':'Medium','q':1000},{'name':'Heavy','q':10000}]:
    c = estimate_rag_costs(s['q'])
    print(f"\n📊 {s['name']} ({c['queries']:,} queries):")
    print(f"   embed:    ${c['embed']:.4f}")
    print(f"   llm in:   ${c['llm_in']:.4f}")
    print(f"   llm out:  ${c['llm_out']:.4f}")
    print(f"   TOTAL:    ${c['total']:.4f}  (per-query ${c['per_query']:.6f})")



📊 Light (100 queries):
   embed:    $0.0040
   llm in:   $0.6000
   llm out:  $0.7500
   TOTAL:    $1.3540  (per-query $0.013540)

📊 Medium (1,000 queries):
   embed:    $0.0400
   llm in:   $6.0000
   llm out:  $7.5000
   TOTAL:    $13.5400  (per-query $0.013540)

📊 Heavy (10,000 queries):
   embed:    $0.4000
   llm in:   $60.0000
   llm out:  $75.0000
   TOTAL:    $135.4000  (per-query $0.013540)


## 17. Direct Anthropic API vs Claude-on-Bedrock — Closing Comparison

Same model (`claude-sonnet-4-5`), two delivery channels. The choice usually comes down to compliance and operations, not capability.

| Dimension | Direct Anthropic API (Lab 6.1) | Claude on AWS Bedrock (this lab) |
|---|---|---|
| **Auth / IAM** | Static API key (rotate manually) | IAM roles + STS — fits banking SSO |
| **Network** | Public internet to `api.anthropic.com` | Private VPC + VPC endpoint (PrivateLink) |
| **Data residency** | Anthropic's selected region (US/EU) | The Bedrock region you call (e.g., `us-east-1`, `ap-south-1` when available) |
| **Latency** | Lower (one network hop, no AWS proxy) | Slightly higher (AWS-internal routing); usually a few hundred ms |
| **Pricing** | Anthropic list price | Same per-token rate; AWS marketplace billing on your AWS bill |
| **Logs / audit** | Anthropic dashboard | CloudTrail + CloudWatch — already wired into bank SOC |
| **Provisioned throughput** | Anthropic capacity | Provisioned Throughput SKU (committed TPS) |
| **Available features** | All Anthropic features as released | Bedrock follows Anthropic with a lag (e.g., extended thinking, computer-use) |
| **When to pick** | Prototyping, non-regulated workloads | Banking / financial-services prod, anything needing IAM + VPC |

**Rule of thumb for eClerx clients:** prototype with the direct API for speed; before pilot, port the LLM call to `ChatBedrock` so the same chain runs against Bedrock with no other code change. The interface is identical — only the import line differs.


In [ ]:
# # Same chain, two LLMs — show that swapping is a one-line change.
# from langchain_anthropic import ChatAnthropic

# direct_llm = ChatAnthropic(model='claude-sonnet-4-5', temperature=0.1, max_tokens=400)

# rag_chain_direct = RunnableParallel(
#     answer=(
#         {'context':  retriever | RunnableLambda(format_docs),
#          'question': RunnablePassthrough()}
#         | qa_prompt | direct_llm | StrOutputParser()
#     ),
#     source_documents=retriever,
# )

# q = 'What is the Tier 1 Capital Ratio in Q3 2024?'

# for label, chain in [('Bedrock', rag_chain), ('Direct API', rag_chain_direct)]:
#     t0 = time.time()
#     out = chain.invoke(q)
#     dt = time.time() - t0
#     print(f'\n--- {label} ({dt:.2f}s) ---')
#     print(out['answer'])


---
## 18. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Bedrock as a delivery channel** | Same Claude model — different IAM, networking, and audit story; pick by compliance, not capability |
| **`BedrockEmbeddings` + Titan v2** | 1024-dim, cosine — the production default for OpenSearch knn indexes |
| **OpenSearch Serverless knn** | HNSW with cosine — scales to millions of chunks; pricing is per OCU, not per query |
| **LCEL RAG with `RunnableParallel`** | Returns answer **and** source documents — no separate retrieve+generate plumbing |
| **Modern memory pattern** | `RunnableWithMessageHistory` replaces the hand-rolled chat-history list — swap storage backends for free |
| **Cost discipline** | Sonnet 4.5 is ~$3/$15 per 1M tokens; pair with a tight `max_tokens` and a 3-doc retriever to keep $/query low |

**Next Lab:** Lab 6.3 — Bedrock Guardrails for PII / PCI-DSS Compliance 🛡️


## 19. Stretch Exercise (Optional)

1. Replace the in-memory history store with `RedisChatMessageHistory` (or `SQLChatMessageHistory`) and re-run the multi-turn cell. The chain code does not change — only `get_history` does.
2. Switch the LLM to `anthropic.claude-haiku-4-5-...` and re-run the cost-monitoring cell. How does $/query change?
3. Add a **filter** to the OpenSearch retriever — return only chunks whose `metadata.filename` starts with `Q3` — and confirm the answer changes.
4. Wrap the RAG chain in `.with_retry(stop_after_attempt=3)` and force a Bedrock throttling exception to see it kick in.
5. Forward-reference: in Module 9 we will trace this chain end-to-end with LangSmith. Add the `LANGSMITH_TRACING=true` env var, re-run, and find the trace in the LangSmith UI.
